## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, HTML

import plotly.io as pio
import plotly.express as px
import pandas as pd

try:
    import google.colab
    pio.renderers.default = 'colab'
except ImportError:
    pio.renderers.default = 'notebook_connected'
print(f"Renderer set to: {pio.renderers.default}")


CONFIG = {
    '3d_sample_size': 500,
    'chart_template': 'plotly_white',
    'random_seed': 42
}

PERSONA_COLORS = {
    "Big Spender": "#2ca02c",      # Hijau (Profit Maker)
    "The Whales": "#1f77b4",   # Biru (Volume/Scale)
    "Quality Seekers": "#ff7f0e",     # Oranye (High Quality)
    "Budget Shoppers": "#d62728",      # Merah (Price Sensitive)
    "Unknown": "#7f7f7f"
}

print("Setup & Configuration Completed (New Personas Updated).")

Renderer set to: colab
Setup & Configuration Completed (New Personas Updated).


## Load Data

In [2]:
try:
    df = pd.read_csv('Hasil_Clustering_Customer.csv')

    required_columns = ['CustomerID', 'TotalQuantity', 'AvgUnitPrice',
                        'AvgTransactionValue', 'Cluster']
    missing_cols = [col for col in required_columns if col not in df.columns]

    if missing_cols:
        raise ValueError(f"CRITICAL ERROR: Kolom berikut hilang dari dataset: {missing_cols}")

    print(f"✅ Data Loaded Successfully: {len(df)} rows.")
    display(df.head())

except FileNotFoundError:
    print("❌ File CSV tidak ditemukan.")
except ValueError as e:
    print(e)

✅ Data Loaded Successfully: 4718 rows.


,CustomerID,TotalQuantity,AvgUnitPrice,AvgTransactionValue,Cluster
0,12004.0,104.0,14.217679,1509.60,2
1,12006.0,2.0,12.380000,24.76,2
2,12008.0,385.0,13.439901,5152.10,0
3,12013.0,3.0,16.304325,69.96,2
4,12024.0,14.0,10.680000,149.52,1


## Dynamic Persona Mapping

In [3]:
cluster_stats = df.groupby('Cluster').agg({
    'AvgTransactionValue': 'mean',
    'TotalQuantity': 'mean',
    'AvgUnitPrice': 'mean'
}).reset_index()

print("--- Statistik Cluster Mentah ---")
display(cluster_stats)


# A. Cari Cluster dengan Nilai Transaksi Tertinggi -> Big Spender
vip_cluster = cluster_stats.loc[cluster_stats['AvgTransactionValue'].idxmax(), 'Cluster']

# B. Sisa cluster selain VIP
remaining = cluster_stats[cluster_stats['Cluster'] != vip_cluster].copy()

# C. Dari sisa, cari yang Quantity-nya Tertinggi -> The Whales
whales_cluster = remaining.loc[remaining['TotalQuantity'].idxmax(), 'Cluster']

# D. Sisa cluster selain Big Spender & Whales
remaining = remaining[remaining['Cluster'] != whales_cluster].copy()

# E. Dari sisa, cari yang Unit Price-nya Tertinggi -> Quality Seekers
quality_cluster = remaining.loc[remaining['AvgUnitPrice'].idxmax(), 'Cluster']

# F. Sisanya -> Budget Shoppers
budget_cluster = remaining[remaining['Cluster'] != quality_cluster]['Cluster'].values[0]

cluster_map = {
    vip_cluster: "Big Spender",
    whales_cluster: "The Whales",
    quality_cluster: "Quality Seekers",
    budget_cluster: "Budget Shoppers"
}

df['Persona'] = df['Cluster'].map(cluster_map)

category_orders = {"Persona": ["Big Spender", "The Whales", "Quality Seekers", "Budget Shoppers"]}

print("\n✅ Mapping Persona Baru Berhasil:")
for k in sorted(cluster_map.keys()):
    print(f"  Cluster {k} -> {cluster_map[k]}")

--- Statistik Cluster Mentah ---


,Cluster,AvgTransactionValue,TotalQuantity,AvgUnitPrice
0,0,3932.500991,657.776675,12.961870
1,1,1223.698162,332.176568,10.949439
2,2,1148.847794,240.886792,13.757336
3,3,2881.921328,1690.700592,12.032701



✅ Mapping Persona Baru Berhasil:
  Cluster 0 -> Big Spender
  Cluster 1 -> Budget Shoppers
  Cluster 2 -> Quality Seekers
  Cluster 3 -> The Whales


## Statistical Validation

In [4]:
# ANOVA (Analysis of Variance)
print("--- Uji Validitas Segmen (ANOVA) ---")

# Apakah Spending Big Spender benar-benar beda signifikan dari yang lain?
f_val, p_val = stats.f_oneway(
    df[df['Persona'] == "Big Spender"]['AvgTransactionValue'],
    df[df['Persona'] == "Budget Shoppers"]['AvgTransactionValue']
)

if p_val < 0.05:
    print(f"✅ Perbedaan Spending antara Big Spender vs Budget Shoppers SIGNIFIKAN (P-value: {p_val:.2e})")
    print("   Artinya: Segmentasi ini valid secara statistik untuk membedakan daya beli.")
else:
    print("⚠️ Perbedaan tidak signifikan. Perlu tuning ulang model clustering.")

--- Uji Validitas Segmen (ANOVA) ---
✅ Perbedaan Spending antara Big Spender vs Budget Shoppers SIGNIFIKAN (P-value: 0.00e+00)
   Artinya: Segmentasi ini valid secara statistik untuk membedakan daya beli.


## Visualization - User vs Revenue

In [5]:
# Agregasi Data
persona_summary = df.groupby('Persona').agg({
    'CustomerID': 'count',
    'AvgTransactionValue': 'sum' # Proxy untuk Total Revenue di konteks snapshot ini
}).reset_index()

persona_summary.columns = ['Persona', 'Jumlah User', 'Est. Total Revenue']

# Create Subplots (Side by Side)
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=['<b>Populasi Pelanggan</b><br>(Berapa banyak orangnya?)',
                                    '<b>Kontribusi Revenue</b><br>(Berapa banyak uangnya?)'])

# Chart 1: Jumlah User
fig.add_trace(go.Pie(labels=persona_summary['Persona'], values=persona_summary['Jumlah User'],
                     name="User Count", hole=0.4,
                     marker_colors=[PERSONA_COLORS[p] for p in persona_summary['Persona']]),
              1, 1)

# Chart 2: Total Revenue
fig.add_trace(go.Pie(labels=persona_summary['Persona'], values=persona_summary['Est. Total Revenue'],
                     name="Revenue", hole=0.4,
                     marker_colors=[PERSONA_COLORS[p] for p in persona_summary['Persona']]),
              1, 2)

fig.update_layout(title_text="<b>Pareto Analysis:</b> Segmen Kecil vs Kontribusi Besar",
                  template=CONFIG['chart_template'])
fig.show()

## Visualization - 3D Profile

In [9]:
sampled_dataFrames = []
target_sample_size = CONFIG['3d_sample_size']

unique_personas = df['Persona'].unique()
for persona in unique_personas:
    subset = df[df['Persona'] == persona]

    n_samples = min(len(subset), target_sample_size)
    subset_sampled = subset.sample(n=n_samples, random_state=42)

    sampled_dataFrames.append(subset_sampled)

df_sampled = pd.concat(sampled_dataFrames, ignore_index=True)

print(f"Data sampling. Total baris: {len(df_sampled)}")

def tetapkan_persona(cluster):
    return cluster_map.get(cluster, "Unknown")

if 'Persona' not in df_sampled.columns:
    print("⚠️ WARNING: Kolom Persona hilang! Memperbaiki...")
    df_sampled['Persona'] = df_sampled['Cluster'].apply(tetapkan_persona)

fig = px.scatter_3d(df_sampled,
                    x='TotalQuantity',
                    y='AvgTransactionValue',
                    z='AvgUnitPrice',
                    color='Persona',
                    color_discrete_map=PERSONA_COLORS, # warna konsisten
                    opacity=0.7,
                    title='<b>Peta Perilaku 3D (Sampled Data)</b><br><sup>Klik & Tahan Mouse untuk Memutar Grafik</sup>',
                    labels={'TotalQuantity': 'Volume Belanja (Qty)',
                            'AvgTransactionValue': 'Nilai Transaksi (IDR)',
                            'AvgUnitPrice': 'Harga Barang (Quality)'})

fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title='Total Quantity',
        yaxis_title='Transaction Value',
        zaxis_title='Unit Price',
        aspectmode='cube'
    ),
    height=600
)

fig.show()

Data sampling. Total baris: 2000


## Business Simulation & Impact

In [7]:
# --- 1. Siapkan Data Baseline per Persona ---
baseline = df.groupby('Persona').agg({
    'CustomerID': 'count',
    'AvgTransactionValue': 'mean',
    'TotalQuantity': 'mean',
    'AvgUnitPrice': 'mean'
}).reset_index()

baseline.columns = ['Persona', 'Jml User', 'Current Avg Tx Value', 'Current Avg Qty', 'Avg Unit Price']

# --- 2. Definisi Skenario Simulasi (Updated Names) ---
simulation_results = []

for index, row in baseline.iterrows():
    persona = row['Persona']
    users = row['Jml User']
    curr_val = row['Current Avg Tx Value']
    curr_qty = row['Current Avg Qty']
    unit_price = row['Avg Unit Price']

    # Skenario Spesifik per Persona (Nama Baru)
    if persona == "The Whales":
        # Strategi: Bulk Discount (Volume naik 20%, Harga turun 5%)
        metric = "Volume Naik (Bulk)"
        new_qty = curr_qty * 1.20
        new_price = unit_price * 0.95
        new_val = new_qty * new_price
        taktik = "Diskon Grosir 5%"

    elif persona == "Big Spender":
        # Strategi: Cross-Selling / Concierge (Value naik 10%)
        metric = "Basket Size Naik"
        new_val = curr_val * 1.10
        taktik = "Layanan Prioritas & Cross-sell"

    elif persona == "Quality Seekers":
        # Strategi: Bundling (Qty naik 15%)
        metric = "Bundling Effect"
        new_qty = curr_qty * 1.15
        new_val = new_qty * unit_price
        taktik = "Product Bundling"

    elif persona == "Budget Shoppers":
        # Strategi: Minimum Spend Threshold (Uplift 15%, Conversion 20%)
        metric = "Upselling (Conv. 20%)"
        uplift = curr_val * 1.15
        new_val = (uplift * 0.2) + (curr_val * 0.8)
        taktik = "Voucher Min. Belanja"

    else:
        continue

    # Hitung Impact
    est_revenue_gain_per_user = new_val - curr_val
    total_potential_gain = est_revenue_gain_per_user * users

    simulation_results.append({
        'Persona': persona,
        'Strategi': taktik,
        'Asumsi Dampak': metric,
        'Current Avg Value': curr_val,
        'Projected Avg Value': new_val,
        'Uplift (%)': ((new_val - curr_val) / curr_val) * 100,
        'Est. Total Revenue Gain': total_potential_gain
    })

# --- 3. Tampilkan Hasil ---
df_sim = pd.DataFrame(simulation_results)

print("📊 HASIL SIMULASI BISNIS")
print("=======================================")
display(df_sim.style.format({
    'Current Avg Value': '$ {:,.2f}',
    'Projected Avg Value': '$ {:,.2f}',
    'Uplift (%)': '{:.1f}%',
    'Est. Total Revenue Gain': '$ {:,.2f}'
}).background_gradient(subset=['Est. Total Revenue Gain'], cmap='Greens'))

📊 HASIL SIMULASI BISNIS


,Persona,Strategi,Asumsi Dampak,Current Avg Value,Projected Avg Value,Uplift (%),Est. Total Revenue Gain
0,Big Spender,Layanan Prioritas & Cross-sell,Basket Size Naik,"$ 3,932.50","$ 4,325.75",10.0%,"$ 316,959.58"
1,Budget Shoppers,Voucher Min. Belanja,Upselling (Conv. 20%),"$ 1,223.70","$ 1,260.41",3.0%,"$ 44,493.67"
2,Quality Seekers,Product Bundling,Bundling Effect,"$ 1,148.85","$ 3,811.05",231.7%,"$ 4,938,393.89"
3,The Whales,Diskon Grosir 5%,Volume Naik (Bulk),"$ 2,881.92","$ 23,191.81",704.7%,"$ 17,161,857.05"


## Analisa Bisnis & Rekomendasi Strategis

In [8]:
from IPython.display import display, HTML

# 1. Hitung Metrik Kunci
total_potential_revenue = df_sim['Est. Total Revenue Gain'].sum()

# 2. Pemetaan Insight Strategis Profesional
insights_map = {
    "Big Spender": "Segmen bernilai tinggi dengan sensitivitas harga rendah. Fokus strategis pada layanan premium, pengalaman personal, dan penawaran eksklusif untuk memaksimalkan Customer Lifetime Value (CLV).",
    "The Whales": "Segmen berbasis volume yang kritis untuk perputaran inventori. Memerlukan struktur harga grosir kompetitif dan insentif pembelian massal untuk mempertahankan pangsa pasar dan margin reseller yang sehat.",
    "Quality Seekers": "Segmen yang sadar kualitas dengan potensi loyalitas merek yang kuat. Optimal untuk upselling produk premium, bundel produk kurasi, dan model pendapatan berbasis langganan.",
    "Budget Shoppers": "Segmen sensitif harga yang memerlukan taktik pembangunan keranjang strategis. Fokus pada ambang batas nilai pesanan minimum (MOV), hook promosi, dan optimalisasi persepsi nilai."
}

# 3. Definisi Segmen dengan Konteks Bisnis
persona_definitions = [
    ("Big Spender", "Pelanggan high-net-worth yang mewakili tier tertinggi dari Nilai Transaksi Rata-rata (ATV). Segmen ini menghasilkan kontribusi pendapatan yang tidak proporsional dan berfungsi sebagai pusat profit utama."),
    ("The Whales", "Segmen B2B dan reseller yang dicirikan oleh transaksi volume tinggi dengan margin rendah. Esensial untuk mempertahankan skala operasional dan penetrasi pasar melalui saluran distribusi."),
    ("Quality Seekers", "Pembeli selektif dengan preferensi pada SKU premium dan kualitas produk superior. Menunjukkan potensi kuat untuk ekspansi kategori dan adopsi produk margin tinggi."),
    ("Budget Shoppers", "Segmen sadar biaya dengan elastisitas harga tinggi. Mewakili peluang pertumbuhan volume melalui mekanik promosi strategis dan strategi optimalisasi keranjang.")
]

# 4. Bangun Tabel Rekomendasi Strategis
table_rows = ""
for index, row in df_sim.iterrows():
    persona = row['Persona']
    strategi = str(row['Strategi']).strip()
    reason = insights_map.get(persona, "-")
    revenue_gain = row['Est. Total Revenue Gain']

    table_rows += f"""
        <tr style="border-bottom: 1px solid #e0e0e0;">
            <td style="padding: 16px; border-right: 1px solid #e0e0e0;">
                <strong style="color: #1a5490; font-size: 15px;">{persona}</strong>
            </td>
            <td style="padding: 16px; border-right: 1px solid #e0e0e0; font-size: 14px;">
                {strategi}
            </td>
            <td style="padding: 16px; border-right: 1px solid #e0e0e0; font-size: 14px; color: #424242;">
                {reason}
            </td>
            <td style="padding: 16px; text-align: right; font-size: 15px;">
                <strong style="color: #2e7d32;">$ {revenue_gain:,.2f}</strong>
            </td>
        </tr>
    """

# 5. Bangun Bagian Definisi Persona
persona_html = ""
for name, description in persona_definitions:
    persona_html += f"""
        <div style="margin-bottom: 18px; padding-left: 20px; border-left: 4px solid #3498db;">
            <strong style="color: #1a5490; font-size: 16px;">{name}</strong>
            <p style="margin: 8px 0 0 0; color: #555; line-height: 1.6; font-size: 15px;">{description}</p>
        </div>
    """

# 6. Laporan HTML Profesional
report_html = f"""
<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 1400px; margin: 30px auto; background: #ffffff; padding: 40px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

    <!-- Header Section -->
    <div style="border-bottom: 4px solid #1a5490; padding-bottom: 20px; margin-bottom: 30px;">
        <h1 style="color: #1a5490; margin: 0; font-size: 32px; font-weight: 600;">
            📊 Ringkasan Eksekutif: Strategi Segmentasi Pelanggan
        </h1>
        <p style="color: #666; margin: 10px 0 0 0; font-size: 14px;">
            Kerangka Kerja Optimalisasi Pendapatan Berbasis Data
        </p>
    </div>

    <!-- Strategic Overview -->
    <div style="background: #f8f9fa; padding: 25px; border-radius: 8px; margin-bottom: 30px;">
        <h2 style="color: #2c3e50; margin: 0 0 15px 0; font-size: 22px;">Sinopsis Eksekutif</h2>
        <ul style="line-height: 2; font-size: 15px; color: #424242; margin: 0; padding-left: 25px;">
            <li><strong>Heterogenitas Pasar:</strong> Analisis mengungkapkan varians signifikan dalam penggerak nilai pelanggan lintas segmen. Pendekatan strategi yang terdiferensiasi diperlukan untuk memaksimalkan kinerja portofolio dibandingkan protokol perlakuan seragam.</li>
            <li><strong>Prioritisasi Strategis:</strong> Alokasi sumber daya harus memprioritaskan inisiatif retensi dan ekspansi untuk segmen <strong>Big Spender</strong> bersama program optimalisasi volume untuk <strong>Juragan Grosir</strong> guna mencapai Return on Marketing Investment (ROMI) yang optimal.</li>
            <li><strong>Pengungkit Pertumbuhan:</strong> Implementasi strategi keterlibatan spesifik segmen menghadirkan peluang peningkatan pendapatan terukur sebesar <strong> $ {total_potential_revenue:,.2f}</strong> berdasarkan basis pelanggan saat ini dan pola perilaku.</li>
        </ul>
    </div>

    <!-- Segment Definitions -->
    <div style="margin-bottom: 35px;">
        <h2 style="color: #2c3e50; margin: 0 0 20px 0; font-size: 22px;">1. Taksonomi Segmen Pelanggan</h2>
        {persona_html}
    </div>

    <hr style="border: none; border-top: 2px solid #e0e0e0; margin: 40px 0;">

    <!-- Strategic Recommendations -->
    <div style="margin-bottom: 35px;">
        <h2 style="color: #2c3e50; margin: 0 0 15px 0; font-size: 22px;">2. Rekomendasi Strategis & Analisis Dampak Pendapatan</h2>
        <p style="color: #666; margin-bottom: 25px; font-size: 15px; line-height: 1.6;">
            Matriks berikut menguraikan strategi intervensi prioritas dengan proyeksi dampak finansial berdasarkan pemodelan perilaku spesifik segmen dan data konversi historis:
        </p>

        <div style="overflow-x: auto;">
            <table style="width: 100%; border-collapse: collapse; background: white; box-shadow: 0 2px 4px rgba(0,0,0,0.08); border-radius: 8px; overflow: hidden;">
                <thead>
                    <tr style="background: linear-gradient(135deg, #1a5490 0%, #2e7d32 100%); color: white;">
                        <th style="padding: 18px; text-align: left; font-weight: 600; font-size: 14px; text-transform: uppercase; letter-spacing: 0.5px;">Segmen Target</th>
                        <th style="padding: 18px; text-align: left; font-weight: 600; font-size: 14px; text-transform: uppercase; letter-spacing: 0.5px;">Strategi yang Direkomendasikan</th>
                        <th style="padding: 18px; text-align: left; font-weight: 600; font-size: 14px; text-transform: uppercase; letter-spacing: 0.5px;">Rasional Strategis</th>
                        <th style="padding: 18px; text-align: right; font-weight: 600; font-size: 14px; text-transform: uppercase; letter-spacing: 0.5px;">Proyeksi Peningkatan Pendapatan</th>
                    </tr>
                </thead>
                <tbody>
                    {table_rows}
                </tbody>
            </table>
        </div>
    </div>

    <hr style="border: none; border-top: 2px solid #e0e0e0; margin: 40px 0;">

    <!-- Financial Impact Summary -->
    <div style="background: linear-gradient(135deg, #2e7d32 0%, #1b5e20 100%); color: white; padding: 35px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(46, 125, 50, 0.3);">
        <div style="font-size: 14px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.9; margin-bottom: 10px;">
            Peluang Pendapatan Agregat
        </div>
        <div style="font-size: 42px; font-weight: 900; margin-bottom: 10px;">
            $ {total_potential_revenue:,.2f}
        </div>
        <div style="font-size: 16px; opacity: 0.95;">
             Total Proyeksi Peningkatan Pendapatan
        </div>
    </div>

    <!-- Footnotes -->
    <div style="margin-top: 30px; padding: 20px; background: #f8f9fa; border-left: 4px solid #ff9800; border-radius: 4px;">
        <p style="margin: 0; color: #666; font-size: 13px; line-height: 1.6;">
            <strong>Catatan Metodologi:</strong> Proyeksi pendapatan diturunkan dari pemodelan perilaku tingkat segmen, metrik konversi historis, dan analisis nilai transaksi rata-rata (ATV).
            Hasil aktual dapat bervariasi berdasarkan efektivitas eksekusi, kondisi pasar, dan dinamika kompetitif.
            Pendekatan implementasi yang direkomendasikan mencakup protokol A/B testing dan siklus optimalisasi iteratif.
        </p>
    </div>

</div>
"""

display(HTML(report_html))

Segmen Target,Strategi yang Direkomendasikan,Rasional Strategis,Proyeksi Peningkatan Pendapatan
Big Spender,Layanan Prioritas & Cross-sell,"Segmen bernilai tinggi dengan sensitivitas harga rendah. Fokus strategis pada layanan premium, pengalaman personal, dan penawaran eksklusif untuk memaksimalkan Customer Lifetime Value (CLV).","$ 316,959.58"
Budget Shoppers,Voucher Min. Belanja,"Segmen sensitif harga yang memerlukan taktik pembangunan keranjang strategis. Fokus pada ambang batas nilai pesanan minimum (MOV), hook promosi, dan optimalisasi persepsi nilai.","$ 44,493.67"
Quality Seekers,Product Bundling,"Segmen yang sadar kualitas dengan potensi loyalitas merek yang kuat. Optimal untuk upselling produk premium, bundel produk kurasi, dan model pendapatan berbasis langganan.","$ 4,938,393.89"
The Whales,Diskon Grosir 5%,Segmen berbasis volume yang kritis untuk perputaran inventori. Memerlukan struktur harga grosir kompetitif dan insentif pembelian massal untuk mempertahankan pangsa pasar dan margin reseller yang sehat.,"$ 17,161,857.05"
